# 03. Expected Power Prediction Modeling

This notebook evaluates **Random Forest** and **XGBoost** regression models to predict expected solar PV power generation based on solar irradiance and temporal parameters.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.append(str(Path("..").resolve()))

from src.config import CLEANED_DATA_PATH, RESULTS_DIR
from src.data_loader import load_nise_raw, extract_key_metrics
from src.preprocessing import clean_nise_data
from src.train import train_models
from src.predict import predict_expected_power

sns.set_theme(style="whitegrid")

## 1. Load Data & Train Models

In [ ]:
df_clean = pd.read_csv(CLEANED_DATA_PATH)
model_rf, model_xgb, metrics_df, X_test, y_test = train_models(df_clean)

## 2. Model Evaluation Summary

In [ ]:
print("--- PERFORMANCE METRICS --- ")
print(metrics_df.to_string(index=False))

## 3. Generate Expected Power Predictions Across Full Dataset

In [ ]:
df_results = predict_expected_power(df_clean)

# Compute Residuals
df_results['residual_kw'] = df_results['total_ac_power_kw'] - df_results['expected_power_kw']

print("Sample Predictions:")
print(df_results[['timestamp', 'irradiance', 'total_ac_power_kw', 'expected_power_kw', 'residual_kw']].head(10))

## 4. Actual vs. Expected Power Visualization

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df_results['timestamp'][:300], df_results['total_ac_power_kw'][:300], label='Actual Power (kW)', color='#2980b9', alpha=0.85)
plt.plot(df_results['timestamp'][:300], df_results['expected_power_kw'][:300], label='Expected Power (XGBoost)', color='#e74c3c', linestyle='--', linewidth=1.5)
plt.xlabel('Timestamp')
plt.ylabel('Power (kW)')
plt.title('Actual vs Expected Solar PV Power Output (First 300 Timesteps)')
plt.legend()
plt.tight_layout()
plt.show()